In [1]:
import pandas as pd
import numpy as np

In [85]:
data = []

# Day 1
day1 = [
    [35.7, 38.4, 34.9, 37.1],
    [37.1, 37.2, 34.3, 35.5],
    [36.7, 38.1, 34.5, 36.5],
    [37.7, 36.9, 33.7, 36.0],
    [35.3, 37.2, 36.2, 33.8],
]

# Day 2
day2 = [
    [34.7, 36.9, 32.0, 35.8],
    [35.2, 38.5, 35.2, 32.9],
    [34.6, 36.4, 33.5, 35.7],
    [36.4, 37.8, 32.9, 38.0],
    [35.2, 36.1, 33.3, 36.1],
]

# データ格納
for day_label, matrix in zip([1, 2], [day1, day2]):
    for row in matrix:
        for worker, value in enumerate(row, start=1):
            data.append([day_label, worker, value])

df = pd.DataFrame(data, columns=["Day", "Worker", "Weight"])

In [170]:
# 目的変数
y = df["Weight"].values

# 交互作用項まで作成
df_S = pd.get_dummies(df, columns = ["Day", "Worker"], dtype = int, drop_first = True)
df_S = df_S.drop(["Weight"], axis = 1)
cols = ["Worker_2", "Worker_3", "Worker_4"]
df_S[[f"Day_2_{c}" for c in cols]] = df_S["Day_2"].values[:, None] * df_S[cols]

# 各仮説ごとに使用しない列
drop_list_S = []
drop_list_I = ["Day_2_Worker_2", "Day_2_Worker_3", "Day_2_Worker_4"]
drop_list_A = ["Day_2", "Day_2_Worker_2", "Day_2_Worker_3", "Day_2_Worker_4"]
drop_list_B = ["Worker_2", "Worker_3", "Worker_4", "Day_2_Worker_2", "Day_2_Worker_3", "Day_2_Worker_4"]
drop_list_M = ['Day_2', 'Worker_2', 'Worker_3', 'Worker_4', 'Day_2_Worker_2','Day_2_Worker_3', 'Day_2_Worker_4']

list_dof = []
list_D = []
for drop_list in [drop_list_S, drop_list_I, drop_list_A, drop_list_B, drop_list_M]:
    
    # 各仮説の説明変数
    X = df_S.drop(drop_list, axis = 1).values
    X = np.column_stack([X,np.ones(len(y))])

    # パラメータ推定
    beta = np.linalg.inv(X.T@X) @ X.T @ y
    
    # 尺度付き逸脱度
    D = y @ y - beta.T @ X.T @y
    
    # 自由度
    dof = X.shape[0] - X.shape[1]

    list_dof.append(dof)
    list_D .append(D)

df_describe = pd.DataFrame(
    {
        "仮説":["S", "I", "A", "B", "M"],
        "自由度":list_dof,
        "尺度付き逸脱度":list_D
    }
)

In [171]:
df_describe

,仮説,自由度,尺度付き逸脱度
0,S,32,40.196
1,I,35,43.154
2,A,36,49.238
3,B,38,97.776
4,M,39,103.860


In [177]:
# 分散分析表の作成

# 自由度
list_dof_anova = [

    # 平均
    len(y) - 39,

    # 工員さ
    38 - 35,

    # 日間差
    36 - 35,
    
    # 交互作用項
    35 - 32,

    # 残差
    len(y) - (len(y) - 39 + 36 - 35 + 38 - 35 + 35 - 32),

    # 合計
    len(y)
]

# sum of squares
list_sum_of_sq_anova = [
    
    # 平均
    y.sum()**2 / len(y),

    # 工員差
    97.776 - 43.154,

    # 日間差
    49.238 - 43.154,

    # 交互作用項
    43.154 - 40.196,

    # 残差
    (y**2).sum() - (y.sum()**2 / len(y) + 49.238 - 43.154 + 97.776 - 43.154 + 43.154 - 40.196),
    
    # 合計
    (y**2).sum() 
    
]
df_anova = pd.DataFrame(
    {
    "自由度":list_dof_anova,
    "平方和":list_sum_of_sq_anova
    },
    index = ["平均", "工員差", "日間差", "交互作用", "残差"] + ["合計"]
)

df_anova["平均平方"] = df_anova["平方和"] / df_anova["自由度"]

# 工員差のF値
F_woker = 18.2073333 / 1.256125

# 日間さのF値
F_day = 6.084 / 1.256125

# 交互作用のF値
F_interaction = 0.986 / 1.256125

df_anova["F値"] = ["-", F_woker, F_day, F_interaction, "-", "-"]
df_anova["上位5%点（自由度(3, 32)の場合）"] = ["-", 2.90, "-", 2.90, "-", "-"]
df_anova["上位5%点（自由度(1, 32)の場合）"] = ["-", "-", 4.15,  "-","-", "-"]

In [178]:
df_anova

,自由度,平方和,平均平方,F値,"上位5%点（自由度(3, 32)の場合）","上位5%点（自由度(1, 32)の場合）"
平均,1,51122.500,51122.500000,-,-,-
工員差,3,54.622,18.207333,14.494842,2.9,-
日間差,1,6.084,6.084000,4.843467,-,4.15
交互作用,3,2.958,0.986000,0.784954,2.9,-
残差,32,40.196,1.256125,-,-,-
合計,40,51226.360,1280.659000,-,-,-


In [161]:
F_woker, F_day, F_interaction

(14.49484191461837, 4.843467011642949, 0.784953726738979)